RAG application for pdf and text files


In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 36.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install transformers sentence-transformers faiss-cpu PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.8 MB/s eta 0:00:00


following is lib for pdf reading


In [ ]:
import PyPDF2

def read_file(file_name):
    if file_name.endswith(".pdf"):
        text = ""
        with open(file_name, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text += page.extract_text()
        return text

    elif file_name.endswith(".txt"):
        with open(file_name, "r", encoding="utf-8") as f:
            return f.read()

In [ ]:
def chunk_text(text, chunk_size=300):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))

    return chunks

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

following is the vector database that i added and document processing function

In [ ]:
import faiss
import numpy as np

In [ ]:
import faiss
import numpy as np

# Global variables to store the processed document state
current_processed_chunks = []
current_faiss_index = None
last_uploaded_file_path = None

# Helper function to encapsulate the Q&A logic after document is processed
def _get_answer_from_processed_document(query_text):
    global current_processed_chunks
    global current_faiss_index
    # The rest (embedder, tokenizer, model) are already global in scope from previous cells.

    if current_faiss_index is None or not current_processed_chunks:
        return "Error: No document has been processed. Please upload a document first."

    query_vec = embedder.encode([query_text])

    # Search the FAISS index
    D, I = current_faiss_index.search(np.array(query_vec), k=3)
    context = "\n".join([current_processed_chunks[i] for i in I[0]])

    # Increase context truncation to provide more information to the model (approx 2048 characters for 512 tokens)
    context = context[:2048] # Truncate context to avoid excessively long prompts

    print("\n--- Retrieved Context ---")
    print(context)
    print("-------------------------\n")

    prompt = f"""
    Context: {context}

    Question: {query_text}

    Answer in a clear and short way:
    """

    print("\n--- Full Prompt Sent to Model ---")
    print(prompt)
    print("----------------------------------\n")

    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=100, # This controls the length of the generated answer
        num_beams=4, # Use beam search for better quality
        early_stopping=True
    )
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return generated_text


def process_and_query(file_obj, query_text):
    global current_processed_chunks
    global current_faiss_index
    global last_uploaded_file_path

    response_message = ""

    # Check if a new file is uploaded or if no file has been processed yet
    if file_obj is not None and file_obj.name != last_uploaded_file_path:
        last_uploaded_file_path = file_obj.name

        # Process the new file
        try:
            text = read_file(last_uploaded_file_path)
            if not text:
                return "Error: Could not read content from the uploaded file. Please try again or upload a different file."

            current_processed_chunks = chunk_text(text)
            if not current_processed_chunks:
                return "Error: Document is empty or could not be chunked. Please check the document content."

            chunk_embeddings = embedder.encode(current_processed_chunks)
            dimension = chunk_embeddings.shape[1]
            current_faiss_index = faiss.IndexFlatL2(dimension)
            current_faiss_index.add(np.array(chunk_embeddings))
            response_message += "Document processed successfully. "
        except Exception as e:
            return f"Error processing document: {e}. Please ensure it's a valid PDF or TXT file."

    # If no document is processed yet, and no file was just uploaded
    if current_faiss_index is None or not current_processed_chunks:
        return "Please upload a document to begin."

    # If query is provided, get an answer
    if query_text:
        answer = _get_answer_from_processed_document(query_text)
        return response_message + "Answer: " + answer
    else:
        return response_message + "Please enter your question."

following is fronted interface

In [ ]:
!pip install -q gradio

In [22]:
import gradio as gr

iface = gr.Interface(
    fn=process_and_query, # Use the new function
    inputs=[
        gr.File(label="Upload Document (PDF/TXT)"),
        gr.Textbox(lines=2, placeholder="Enter your question here...")
    ],
    outputs=gr.Textbox(lines=(5, 20), label="Answer", autoscroll=True),
    title="Document Q&A System",
    description="Upload a PDF/TXT document, then ask questions about its content."
)

iface.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bac68428c47646654b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error


--- Retrieved Context ---
a whole weekend to studying, you just discovered that you failed a test in your psychology class. OPT iON A: Ask a student to role-play each of the different reactions and ask the students’ classmates to identify which defense mechanism the student is attempting to portray. Simply cut the worksheet into strips and give each “actor” one of the reactions. Y ou will also need a student to play the role of the teacher passing back the tests. OPT iON B: Give a copy of the worksheet (on next page) to each student to complete independently.BACK TO CONTENT OUTLINE54 PERSONALITY BACK TO CONTENTSWORKSHEET A. Denial B. Displacement C. intellectualization D. Projection E. Rationalization F. Reaction formation G. Regression H. Repression i . Sublimation 1. ____ When the teacher hands y ou the test you failed, you honestly ex- claim, “I don’t even remember taking the test!” 2. ____ When the teacher hands y ou the test you failed, you hand it back and say, “This can’t be mi